In [ ]:
import xarray as xr

ds = xr.open_dataset(r"Data/raster/drought_indices/spi/SPI_1991.nc")
print(ds)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

# === Load the shapefile ===
shapefile_path = r'Eastern_Aus_Bunbury/Eastern_Aus_Bunbury.shp'
qld_shape = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# === Open the SPI dataset ===
nc_path = Path(r'Data/raster/drought_indices/spi/SPI_2022.nc')
year = nc_path.stem.split('_')[1]

ds = xr.open_dataset(nc_path)

# === Select SPI variable ONCE ===
var_name = 'spi_1'          # change here: spi_3, spi_6, etc.
monthly_rain = ds[var_name]

# === Convert variable name to display label ===
spi_label = var_name.replace('_', '-').upper()   # spi_1 → SPI-1

# === Fix metadata so colorbar label is correct ===
monthly_rain.attrs.pop('units', None)
monthly_rain.attrs['long_name'] = spi_label

# === Create figure and axes ===
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
axes = axes.flatten()

# === Plot each month ===
for i, month in enumerate(monthly_rain.time):
    ax = axes[i]
    data = monthly_rain.sel(time=month)

    data.plot(
        ax=ax,
        cmap='RdBu',
        add_colorbar=True
    )

    # Overlay region boundaries
    qld_shape.boundary.plot(
        ax=ax,
        color='black',
        linewidth=0.5,
        alpha=0.7
    )

    # Titles and formatting
    ax.set_title(month.dt.strftime('%B').item(), fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')

# === Remove unused subplots ===
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# === Figure title (automatic) ===
plt.suptitle(
    f'Monthly {spi_label} for {year} in WheatBelt',
    fontsize=16,
    y=0.98
)

plt.tight_layout()
plt.show()


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

# === Paths ===
data_dir = Path(r'Data/raster/drought_indices/spi')
output_dir = Path(r'Paper/Figures/Drought/SPI')
output_dir.mkdir(parents=True, exist_ok=True)

shapefile_path = r'Eastern_Aus_Bunbury/Eastern_Aus_Bunbury.shp'
qld_shape = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# === Years and SPI variables ===
years = range(1991, 2024)
spi_vars = ['spi_1', 'spi_3', 'spi_6', 'spi_12']

# === Loop over years ===
for year in years:
    nc_path = data_dir / f'SPI_{year}.nc'

    if not nc_path.exists():
        print(f"Missing file: {nc_path}")
        continue

    ds = xr.open_dataset(nc_path)

    # === Loop over SPI scales ===
    for var_name in spi_vars:

        if var_name not in ds:
            print(f"{var_name} not found in {nc_path.name}")
            continue

        monthly_rain = ds[var_name]

        # === Create display label ===
        spi_label = var_name.replace('_', '-').upper()   # spi_1 → SPI-1
        file_label = spi_label.replace('-', '')          # SPI-1 → SPI1

        # === Fix metadata ===
        monthly_rain.attrs.pop('units', None)
        monthly_rain.attrs['long_name'] = spi_label

        # === Create figure ===
        fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
        axes = axes.flatten()

        # === Plot each month ===
        for i, month in enumerate(monthly_rain.time):
            ax = axes[i]
            data = monthly_rain.sel(time=month)

            data.plot(
                ax=ax,
                cmap='RdBu',
                add_colorbar=True
            )

            qld_shape.boundary.plot(
                ax=ax,
                color='black',
                linewidth=0.5,
                alpha=0.7
            )

            ax.set_title(month.dt.strftime('%B').item(), fontsize=11)
            ax.set_xlabel('')
            ax.set_ylabel('')

        # === Remove unused subplots ===
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        # === Suptitle ===
        plt.suptitle(
            f'Monthly {spi_label} for {year} in WheatBelt',
            fontsize=16,
            y=0.98
        )

        plt.tight_layout()

        # === Save figure ===
        output_file = output_dir / f'{file_label}_{year}.png'
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Saved: {output_file.name}")

    ds.close()
